In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *

In [55]:
#files = sorted(glob.glob('/home/ulyanov/data/solo/phi/nonpolar/*'))
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/nonpolar/*'))
print(len(files))
print(files[0], files[-1])

357
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_000000_TAI.3.magnetogram.fits /home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241130_220000_TAI.3.magnetogram.fits


In [62]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 512

view_north = View(nx, ny, xc, yc, Rsun, -90, 90, 0) ### North pole view
grid = np.mgrid[:nx,:ny].astype(np.float32)

mean, variance, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

plt.ioff()

for file in files[:]:
    print(file)

    with fits.open(file) as hdul:
        header = hdul[1].header.copy()
        data = hdul[1].data.copy()

    #data = rebin(data, 4, update_header=header)
    view = View.from_header(header)

    transform = view_north.to_spherical(correct_mu=True, mu_thr=0.1) - view.to_spherical(correct_mu=True, correct_dr=True, mu_thr=0.1)
    grid_, alpha = transform(grid)
    data = bilinear(data, *grid_) * alpha

    n = (~np.isnan(data)) * (1 / alpha) ** 4
    coverage += np.nan_to_num(n)
    mean_ = mean.copy()
    mean += np.nan_to_num((data - mean) * n / coverage)
    variance += np.nan_to_num((data - mean) * (data - mean_) * n)

    #show_polar_data(mean, vmin=-20, vmax=20, cmap='seismic', label=r'$B_{los}, G$')
    #plt.savefig(f'temp/{file.split("/")[-1].split(".")[0]}.png')
    #plt.savefig(f'temp/{file.split("/")[-1].split(".")[2]}.png')
    #plt.close()

plt.ion()

variance = variance / coverage
mean[coverage == 0] = np.nan
coverage[coverage == 0] = np.nan
variance[coverage == 0] = np.nan

/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_000000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_020000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_040000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_060000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_080000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_100000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_120000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_140000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_160000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_180000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.20241101_200000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/nonpolar/hmi.M_720s.2024110

/tmp/ipykernel_17091/2357200021.py:39: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [60]:
show_polar_data(mean, vmin=-20, vmax=20, cmap='seismic', label=r'$B_{los}, G$')

(<Figure size 1000x1000 with 2 Axes>, <Axes: >)

In [37]:
show_polar_data(np.sqrt(variance), vmin=0, vmax=20, cmap='turbo', label=r'$\sigma_B, G$')

(<Figure size 1000x1000 with 2 Axes>, <Axes: >)

In [63]:
plt.figure(figsize=(8,8))
plt.plot(mean, mean_fdt, '.', ms=1, color='tab:blue')

plt.xlabel('HMI, G')
plt.ylabel('FDT, G')

plt.xlim(-100,100)
plt.ylim(-100,100)

plt.grid(True)
plt.tight_layout()

In [54]:
#mean_fdt = mean.copy()